In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from langchain_core.tools import BaseTool
import neo4j
from pydantic import BaseModel, Field

from neo4j_graphrag.llm import LLMInterface
from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from app.core.config import settings
from qdrant_client import QdrantClient
from neo4j_graphrag.retrievers import QdrantNeo4jRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OpenAILLM as LLM
from qdrant_client.http.models import VectorParams
from qdrant_client.http.models.models import Distance
from pydantic import BaseModel
from typing import Dict, Any

In [2]:
embedder = SentenceTransformerEmbeddings(model="all-MiniLM-L6-v2")


/Users/sverma22/Projects/oracle-mas/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

client = QdrantClient(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
)

existing = [c.name for c in client.get_collections().collections]
if settings.qdrant_collection_name not in existing:
    client.create_collection(
        collection_name=settings.qdrant_collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
existing

['my-collection']

In [4]:
from langchain_aws import ChatBedrock


In [31]:

from langchain_core.messages import AIMessage
import json

class ArbitraryJson(BaseModel):
    """A model to capture any arbitrary JSON object."""
    data: Dict[str, Any]
    
class BedrockLLM(LLMInterface):
    def __init__(self, model_name: str, **kwargs):
        super().__init__(model_name=model_name, **kwargs)
        self.llm = ChatBedrock(
            model_id=model_name,
            **kwargs
        ).with_structured_output(schema=ArbitraryJson)
    def invoke(self, prompt: str, **kwargs) -> str:
        res : ArbitraryJson = self.llm.invoke(prompt, **kwargs)
        json_str = json.dumps(res.data)
        return AIMessage(content=json_str)
    async def ainvoke(self, prompt: str, **kwargs) -> str:
        res : ArbitraryJson = await self.llm.ainvoke(prompt, **kwargs)
        json_str = json.dumps(res.data)
        return AIMessage(content=json_str)


In [32]:

# llm=LLM(
#     model_name="bedrock/openai.gpt-oss-20b-1:0",
#     model_params={
#        "temperature": 0,
#        "response_format": {"type": "json_schema", "schema": {}}
#     },
#     base_url="http://0.0.0.0:4000",
#     api_key="xxxxxxxx",
# )

llm = BedrockLLM(
    model_name="us.amazon.nova-lite-v1:0",
    # region_name="us-east-2",
)

In [33]:

URI = settings.neo4j_uri
AUTH = (settings.neo4j_username, settings.neo4j_password)

neo4j_driver = neo4j.GraphDatabase.driver(
    URI,
    auth=AUTH,
)

retriever = QdrantNeo4jRetriever(
    driver=neo4j_driver,
    client=client,
    collection_name=settings.qdrant_collection_name,
    using=settings.qdrant_using,
    id_property_external="neo4j_id",    # The payload field that contains identifier to a corresponding Neo4j node id property
    id_property_neo4j="id",
    embedder=embedder,
)

kg_builder = SimpleKGPipeline(
    llm=llm, # an LLMInterface for Entity and Relation extraction
    driver=neo4j_driver,  # a neo4j driver to write results to graph
    embedder=embedder,  # an Embedder for chunks
    from_pdf=False,   # set to False if parsing an already extracted text
)


rag = GraphRAG(retriever=retriever, llm=llm)

In [34]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "/Users/sverma22/Downloads/graphrag.pdf"
loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)
single_text = "\n\n".join([page.page_content for page in pages])

In [35]:
await kg_builder.run_async(text=single_text)

ERROR:neo4j_graphrag.experimental.components.entity_relation_extractor:LLM response has improper format for chunk_index=18
ERROR:neo4j_graphrag.experimental.components.entity_relation_extractor:LLM response has improper format for chunk_index=21


ModelErrorException: An error occurred (ModelErrorException) when calling the Converse operation: Model produced invalid sequence as part of ToolUse. Please refer to the model tool use troubleshooting guide.

ERROR:neo4j_graphrag.experimental.components.entity_relation_extractor:LLM response has improper format for chunk_index=19
